# DemandPulse AI - Exploratory Data Analysis

Concise EDA for the Kaggle dataset:
**Retail Store Inventory and Demand Forecasting** (synthetically generated, Apache 2.0).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import matplotlib.pyplot as plt

from data_preprocessing import load_data, dataset_stats

df = load_data()
print(dataset_stats(df))
df.head()

## 1. Overall demand trend (weekly average across all stores/products)

In [ ]:
weekly = df.set_index('Date').resample('W')['Demand'].mean()
ax = weekly.plot(figsize=(11, 3), color='#4f8cff', title='Average weekly demand')
ax.set_ylabel('Demand (units)')
plt.tight_layout()
plt.show()

## 2. Demand trend for one store/product combination

In [ ]:
combo = df[(df['Store ID'] == 'S001') & (df['Product ID'] == 'P0001')]
ax = combo.set_index('Date')['Demand'].rolling(14).mean() \
        .plot(figsize=(11, 3), color='#2ecc8f', title='S001 / P0001 - 14-day rolling demand')
ax.set_ylabel('Demand (units)')
plt.tight_layout()
plt.show()

## 3. Demand distribution

In [ ]:
ax = df['Demand'].plot(kind='hist', bins=50, figsize=(8, 3), color='#f5a623',
                       title='Demand distribution')
ax.set_xlabel('Units')
plt.tight_layout()
plt.show()

## 4. Seasonality: weekday / month / season patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
df.groupby(df['Date'].dt.day_name())['Demand'].mean() \
    .reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']) \
    .plot(kind='bar', ax=axes[0], color='#4f8cff', title='Avg demand by weekday')
df.groupby(df['Date'].dt.month)['Demand'].mean() \
    .plot(kind='bar', ax=axes[1], color='#2ecc8f', title='Avg demand by month')
df.groupby('Seasonality')['Demand'].mean() \
    .plot(kind='bar', ax=axes[2], color='#f5a623', title='Avg demand by season')
for a in axes: a.set_ylabel('Units')
plt.tight_layout()
plt.show()

## 5. Promotion vs demand

In [ ]:
promo = df.groupby('Promotion')['Demand'].mean()
print(promo)
promo.plot(kind='bar', figsize=(5, 3), color=['#8b98b3', '#4f8cff'],
           title='Average demand: no promo vs promo')
plt.ylabel('Units'); plt.tight_layout(); plt.show()

## 6. Inventory level vs demand

In [ ]:
ax = df.plot(kind='scatter', x='Inventory Level', y='Demand', alpha=0.05, figsize=(6, 3.5),
             title='Inventory Level vs Demand')
plt.tight_layout()
plt.show()

## 7. Feature correlations (numeric columns)

In [ ]:
corr = df[['Demand', 'Units Sold', 'Inventory Level', 'Price', 'Discount',
           'Promotion', 'Competitor Pricing', 'Epidemic']].corr()
fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.columns)), corr.columns)
fig.colorbar(im, shrink=0.8)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()
corr['Demand'].sort_values(ascending=False)

## Notes
- Demand is the forecasting target; `Units Ordered` and `Inventory Level` are **never** used
  as predictors (data-leakage policy - see REPORT.md).
- All statistics come from the downloaded CSV at `data/retail_store_inventory.csv`.